# Data Validation & EDA

This notebook validates the three project datasets before modeling:
- Support tickets
- Transactions / fraud data
- QA benchmark pairs


## 1. Setup and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

TICKETS_PATH = PROJECT_ROOT / "data" / "support_tickets.csv"
TRANSACTIONS_PATH = PROJECT_ROOT / "data" / "transactions.csv"
QA_PATH = PROJECT_ROOT / "data" / "qa_pairs.json"

print("Project root:", PROJECT_ROOT)
print("Tickets:", TICKETS_PATH.exists())
print("Transactions:", TRANSACTIONS_PATH.exists())
print("QA pairs:", QA_PATH.exists())

Project root: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System
Tickets: True
Transactions: True
QA pairs: True


## 2. Support Tickets Validation

In [2]:
tickets = pd.read_csv(TICKETS_PATH)

print("Shape:", tickets.shape)
print("\nColumns:")
print(tickets.columns.tolist())

print("\nData types:")
print(tickets.dtypes)

Shape: (200, 14)

Columns:
['ticket_id', 'date_created', 'customer_id', 'channel', 'category', 'sub_category', 'query_text', 'sentiment', 'risk_level', 'resolution_text', 'resolution_time_minutes', 'resolved_by', 'customer_satisfaction', 'escalated']

Data types:
ticket_id                    str
date_created                 str
customer_id                  str
channel                      str
category                     str
sub_category                 str
query_text                   str
sentiment                    str
risk_level                   str
resolution_text              str
resolution_time_minutes    int64
resolved_by                  str
customer_satisfaction      int64
escalated                    str
dtype: object


In [3]:
print("Missing values:")
display(tickets.isnull().sum())

print("Duplicate rows:", tickets.duplicated().sum())
print("Duplicate ticket IDs:", tickets["ticket_id"].duplicated().sum())

Missing values:


ticket_id                  0
date_created               0
customer_id                0
channel                    0
category                   0
sub_category               0
query_text                 0
sentiment                  0
risk_level                 0
resolution_text            0
resolution_time_minutes    0
resolved_by                0
customer_satisfaction      0
escalated                  0
dtype: int64

Duplicate rows: 0
Duplicate ticket IDs: 0


### 2.1 Category, Risk, Sentiment and Escalation

In [4]:
for column in ["category", "sentiment", "risk_level", "channel", "escalated"]:
    print(f"\n{column}:")
    print(tickets[column].value_counts(dropna=False))


category:
category
Fraud/Unauthorized    50
Loan                  50
KYC                   50
Account Access        50
Name: count, dtype: int64

sentiment:
sentiment
Anxious       39
Confused      37
Urgent        33
Neutral       32
Frustrated    31
Angry         28
Name: count, dtype: int64

risk_level:
risk_level
Low       105
Medium     60
High       35
Name: count, dtype: int64

channel:
channel
Email         45
Branch        41
Phone         40
Web Chat      39
Mobile App    35
Name: count, dtype: int64

escalated:
escalated
No     183
Yes     17
Name: count, dtype: int64


In [5]:
print("Category vs Risk:")
display(
    pd.crosstab(
        tickets["category"],
        tickets["risk_level"]
    )
)

print("Category vs Escalation:")
display(
    pd.crosstab(
        tickets["category"],
        tickets["escalated"]
    )
)

Category vs Risk:


risk_level,High,Low,Medium
category,,,
Account Access,0,45,5
Fraud/Unauthorized,30,10,10
KYC,0,20,30
Loan,5,30,15


Category vs Escalation:


escalated,No,Yes
category,,
Account Access,50,0
Fraud/Unauthorized,37,13
KYC,50,0
Loan,46,4


In [6]:
category_summary = (
    tickets.groupby("category")
    .agg(
        avg_resolution_time=("resolution_time_minutes", "mean"),
        avg_customer_satisfaction=("customer_satisfaction", "mean"),
        escalation_rate=(
            "escalated",
            lambda x: (x == "Yes").mean() * 100
        )
    )
    .round(2)
)

display(category_summary)

,avg_resolution_time,avg_customer_satisfaction,escalation_rate
category,,,
Account Access,131.06,4.04,0.0
Fraud/Unauthorized,47.10,3.74,26.0
KYC,84.96,4.12,0.0
Loan,100.00,3.82,8.0


### 2.2 Intent Data Quality

The same query may appear multiple times because the ticket dataset contains repeated customer-query templates. We check whether any exact query maps to multiple intent categories.

In [7]:
unique_query_count = tickets["query_text"].nunique()
duplicate_query_rows = tickets["query_text"].duplicated().sum()

print("Unique query_text:", unique_query_count)
print("Duplicate query rows:", duplicate_query_rows)

query_category_check = (
    tickets.groupby("query_text")["category"]
    .nunique()
)

conflicting_queries = query_category_check[
    query_category_check > 1
]

print("\nQueries mapped to multiple categories:")
display(conflicting_queries)

print(
    "Unique queries with exactly one category:",
    (query_category_check == 1).sum()
)

Unique query_text: 66
Duplicate query rows: 134

Queries mapped to multiple categories:


Series([], Name: category, dtype: int64)

Unique queries with exactly one category: 66


### 2.3 Sentiment Label Consistency

Sentiment is checked separately because the same exact query can have different sentiment labels in this dataset. This determines whether supervised text-only sentiment modeling is appropriate.

In [8]:
query_sentiment_check = (
    tickets.groupby("query_text")["sentiment"]
    .nunique()
)

print("Queries with exactly one sentiment:", (query_sentiment_check == 1).sum())
print("Queries with multiple sentiments:", (query_sentiment_check > 1).sum())

sentiment_agreement = (
    tickets.groupby("query_text")["sentiment"]
    .value_counts(normalize=True)
    .groupby(level=0)
    .max()
)

print("Average dominant-sentiment agreement:", round(sentiment_agreement.mean(), 3))
print("Minimum agreement:", round(sentiment_agreement.min(), 3))
print("Maximum agreement:", round(sentiment_agreement.max(), 3))

Queries with exactly one sentiment: 31
Queries with multiple sentiments: 35
Average dominant-sentiment agreement: 0.721
Minimum agreement: 0.2
Maximum agreement: 1.0


In [9]:
query_sentiment_counts = pd.crosstab(
    tickets["query_text"],
    tickets["sentiment"]
)

best_correct = query_sentiment_counts.max(axis=1).sum()
theoretical_text_only_accuracy = best_correct / len(tickets)

print("Theoretical text-only sentiment accuracy ceiling:", round(theoretical_text_only_accuracy, 3))
print("Perfectly consistent queries:", (
    query_sentiment_counts.max(axis=1) == query_sentiment_counts.sum(axis=1)
).sum())

Theoretical text-only sentiment accuracy ceiling: 0.555
Perfectly consistent queries: 31


### Sentiment conclusion

Because exact queries can map to multiple sentiment labels, sentiment is treated as a **runtime Gemini classification task**, not as a supervised model trained on these ticket labels.

## 3. Transaction / Fraud Data Validation

In [10]:
transactions = pd.read_csv(TRANSACTIONS_PATH)

print("Shape:", transactions.shape)
print("\nColumns:")
print(transactions.columns.tolist())

print("\nData types:")
print(transactions.dtypes)

Shape: (2000, 16)

Columns:
['transaction_id', 'account_id', 'timestamp', 'amount_inr', 'merchant_name', 'merchant_category', 'transaction_type', 'city', 'hour_of_day', 'day_of_week', 'is_international', 'velocity_flag', 'geo_anomaly_flag', 'high_amount_flag', 'fraud_label', 'fraud_reason']

Data types:
transaction_id           str
account_id               str
timestamp                str
amount_inr           float64
merchant_name            str
merchant_category        str
transaction_type         str
city                     str
hour_of_day            int64
day_of_week              str
is_international         str
velocity_flag            str
geo_anomaly_flag         str
high_amount_flag         str
fraud_label            int64
fraud_reason             str
dtype: object


In [11]:
print("Missing values:")
display(transactions.isnull().sum())

print("Duplicate rows:", transactions.duplicated().sum())
print("Duplicate transaction IDs:", transactions["transaction_id"].duplicated().sum())

Missing values:


transaction_id          0
account_id              0
timestamp               0
amount_inr              0
merchant_name           0
merchant_category       0
transaction_type        0
city                    0
hour_of_day             0
day_of_week             0
is_international        0
velocity_flag           0
geo_anomaly_flag        0
high_amount_flag        0
fraud_label             0
fraud_reason         1756
dtype: int64

Duplicate rows: 0
Duplicate transaction IDs: 0


### 3.1 Fraud Target Distribution

In [12]:
target_distribution = pd.DataFrame({
    "count": transactions["fraud_label"].value_counts(),
    "percentage": transactions["fraud_label"].value_counts(normalize=True).mul(100)
}).round(2)

display(target_distribution)

,count,percentage
fraud_label,,
0,1756,87.8
1,244,12.2


In [13]:
display(
    transactions.groupby("fraud_label")["amount_inr"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

,count,mean,median,min,max
fraud_label,,,,,
0,1756,2815.83,1306.43,50.18,14999.40
1,244,24544.40,23887.62,272.28,49979.31


### 3.2 Feature Leakage / Synthetic Shortcut Checks

The goal here is to identify features that may make a model look unrealistically strong. `fraud_reason` is checked separately because it is directly tied to the target outcome.

In [14]:
for column in [
    "is_international",
    "velocity_flag",
    "geo_anomaly_flag",
    "high_amount_flag"
]:
    print(f"\n{column} vs fraud_label:")
    display(pd.crosstab(
        transactions[column],
        transactions["fraud_label"]
    ))


is_international vs fraud_label:


fraud_label,0,1
is_international,,
No,1756,0
Yes,0,244



velocity_flag vs fraud_label:


fraud_label,0,1
velocity_flag,,
No,1756,82
Yes,0,162



geo_anomaly_flag vs fraud_label:


fraud_label,0,1
geo_anomaly_flag,,
No,1756,0
Yes,0,244



high_amount_flag vs fraud_label:


fraud_label,0,1
high_amount_flag,,
No,1756,103
Yes,0,141


In [15]:
print("Fraud reason distribution:")
display(transactions["fraud_reason"].value_counts(dropna=False))

print("Fraud reason vs fraud label:")
display(
    pd.crosstab(
        transactions["fraud_reason"].fillna("No Fraud"),
        transactions["fraud_label"]
    )
)

Fraud reason distribution:


fraud_reason
NaN                 1756
High amount           56
Location anomaly      55
Velocity breach       52
Unknown merchant      44
Off-hours             37
Name: count, dtype: int64

Fraud reason vs fraud label:


fraud_label,0,1
fraud_reason,,
High amount,0,56
Location anomaly,0,55
No Fraud,1756,0
Off-hours,0,37
Unknown merchant,0,44
Velocity breach,0,52


In [16]:
fraud_indicators = (
    (transactions["is_international"] == "Yes").astype(int)
    + (transactions["velocity_flag"] == "Yes").astype(int)
    + (transactions["geo_anomaly_flag"] == "Yes").astype(int)
    + (transactions["high_amount_flag"] == "Yes").astype(int)
)

transactions["fraud_indicator_count"] = fraud_indicators

print("Indicator count vs fraud label:")
display(
    pd.crosstab(
        transactions["fraud_indicator_count"],
        transactions["fraud_label"],
        normalize="index"
    ).round(3)
)

Indicator count vs fraud label:


fraud_label,0,1
fraud_indicator_count,,
0,1.0,0.0
2,0.0,1.0
3,0.0,1.0
4,0.0,1.0


### Fraud modeling conclusion

The dataset contains strong synthetic shortcuts. The final behavioral fraud model therefore uses only:

- `amount_inr`
- `hour_of_day`
- `day_of_week`

Identifiers, the target, and direct-leakage fields are excluded.

## 4. QA Benchmark Validation

In [17]:
qa = pd.read_json(QA_PATH)

print("Shape:", qa.shape)
print("\nColumns:")
print(qa.columns.tolist())

print("\nMissing values:")
display(qa.isnull().sum())

print("Duplicate rows:", qa.duplicated().sum())
print("Duplicate QA IDs:", qa["id"].duplicated().sum())

Shape: (20, 7)

Columns:
['id', 'category', 'question', 'answer', 'policy_ref', 'risk_level', 'suggested_action']

Missing values:


id                  0
category            0
question            0
answer              0
policy_ref          0
risk_level          0
suggested_action    0
dtype: int64

Duplicate rows: 0
Duplicate QA IDs: 0


In [18]:
print("Category distribution:")
display(qa["category"].value_counts())

print("Risk distribution:")
display(qa["risk_level"].value_counts())

print("Sample:")
display(qa.head())

Category distribution:


category
Fraud             6
Loan              6
KYC               5
Account Access    3
Name: count, dtype: int64

Risk distribution:


risk_level
Low       11
High       6
Medium     3
Name: count, dtype: int64

Sample:


,id,category,question,answer,policy_ref,risk_level,suggested_action
0,QA001,Fraud,"I noticed an unauthorized transaction of ₹15,0...",Please report immediately via our 24x7 helplin...,"Fraud Handling Policy §3, §4",High,"Block card, raise dispute, escalate to fraud team"
1,QA002,Fraud,How long does fraud investigation take?,Fraud investigations are completed within 30-4...,Fraud Handling Policy §4,Medium,Update customer with reference number
2,QA003,Fraud,Someone used my OTP to transfer money. Can I g...,"If the OTP was shared voluntarily, liability m...","Fraud Handling Policy §3.2, §6",High,"Suspend account access, escalate to fraud team..."
3,QA004,Fraud,Multiple small transactions are showing up tha...,Multiple micro-transactions could indicate car...,Fraud Handling Policy §5 High Risk,High,"Block card, dispatch new card, file FIR"
4,QA005,Fraud,I received an OTP I didn't request. Is my acco...,An unrequested OTP could indicate a fraud atte...,"Fraud Handling Policy §5, §6.2",High,"Suspend account, advise branch visit, reset cr..."


## 5. Validation Summary

### Key decisions

1. Support-ticket `query_text` is suitable for intent classification because exact queries map to a single category in the dataset.
2. Ticket sentiment labels are inconsistent for repeated queries, so supervised text-only sentiment modeling is avoided.
3. Fraud data contains strong synthetic shortcuts; the final model uses a small behavioral feature set to reduce shortcut dependence.
4. `fraud_reason` is treated as leakage and is not used as a fraud-model feature.
5. QA pairs are retained as a validation benchmark rather than as the primary RAG corpus.

The next notebooks will focus on **intent modeling**, **fraud modeling**, **RAG**, **evaluation**, and **LangGraph/application testing**.